In [0]:
df_path = "/Volumes/workspace/default/data/paysim.csv"

raw_df = spark.read.csv(df_path, header=True, inferSchema=True)
print("Total rows:", raw_df.count())
raw_df.show(5)

In [0]:
from pyspark.sql.functions import current_timestamp, lit, rand
import uuid

raw_df = spark.read.csv("/Volumes/workspace/default/data/paysim.csv", header=True, inferSchema=True)

fraud_df = raw_df.filter("isFraud = 1")
nonfraud_df = raw_df.filter("isFraud = 0")
fraction = (400_000 - fraud_df.count()) / nonfraud_df.count()
nonfraud_sample = nonfraud_df.sample(withReplacement=False, fraction=fraction, seed=42)
incremental_pool_df = nonfraud_df.subtract(nonfraud_sample)

full_load_df = fraud_df.union(nonfraud_sample).orderBy(rand(seed=42))
full_load_df = full_load_df.withColumn("ingestion_timestamp", current_timestamp()).withColumn("source_batch_id", lit("full_load"))
full_load_df.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/data/bronze/transactions")

batch_df = incremental_pool_df.limit(7000)
batch_df = batch_df.withColumn("ingestion_timestamp", current_timestamp()).withColumn("source_batch_id", lit(f"incremental_{uuid.uuid4().hex[:6]}"))
batch_df.write.format("delta").mode("append").save("/Volumes/workspace/default/data/bronze/transactions")

bronze_df = spark.read.format("delta").load("/Volumes/workspace/default/data/bronze/transactions")
print("Total bronze rows:", bronze_df.count())
bronze_df.groupBy("source_batch_id").count().show()

full_load_df.limit(2000).toPandas().to_csv("/Volumes/workspace/default/data/full_load_sample.csv", index=False)
batch_df.limit(500).toPandas().to_csv("/Volumes/workspace/default/data/incremental_load_sample.csv", index=False)
print("Sample files exported")

In [0]:
import shutil, os

repo_path = "/Workspace/Users/ssalmanaali80@gmail.com/Bank-Transactions-Fraud-Analytics/data/samples"
os.makedirs(repo_path, exist_ok=True)

shutil.copy("/Volumes/workspace/default/data/full_load_sample.csv", f"{repo_path}/full_load_sample.csv")
shutil.copy("/Volumes/workspace/default/data/incremental_load_sample.csv", f"{repo_path}/incremental_load_sample.csv")